# Gabarito — Congruência de Triângulos

Este notebook traz a resolução comentada dos exercícios de [`0405a_congruencia_de_triangulos.ipynb`](0405a_congruencia_de_triangulos.ipynb).

⚠️ **Antes de olhar aqui:** tente resolver os exercícios sozinho no notebook original e use as células de verificação (`assert`) para conferir sua resposta. Volte a este gabarito só depois de tentar — ou se travar de verdade em algum passo.

In [1]:
# Imports necessários para os exercícios abaixo.
import math

import numpy as np


# Ferramentas das seções 1 e 2 do notebook principal (reconstruídas aqui).
def direcao(angulo_graus: float) -> np.ndarray:
    """Vetor de tamanho 1 apontando na direção dada, em graus, a partir da horizontal."""
    rad = math.radians(angulo_graus)
    return np.array([math.cos(rad), math.sin(rad)])


def distancia(P, Q) -> float:
    """Distância entre os pontos P e Q do plano."""
    return math.dist(P, Q)


def medir_angulo(vertice, P, Q) -> float:
    """Transferidor digital: medida, em graus, do ângulo com vértice em `vertice` entre P e Q."""
    u = np.subtract(P, vertice)
    v = np.subtract(Q, vertice)
    cosseno = np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))
    return math.degrees(math.acos(np.clip(cosseno, -1, 1)))


def construir_por_angulos(alfa: float, beta: float, base: float = 5.0):
    """Vértices (A, B, C) de um triângulo com AB = base, Â = alfa e B̂ = beta (receita ALA)."""
    if alfa <= 0 or beta <= 0 or alfa + beta >= 180:
        return None
    A, B = np.array([0.0, 0.0]), np.array([float(base), 0.0])
    t, _ = np.linalg.solve(np.column_stack([direcao(alfa), -direcao(180 - beta)]), B - A)
    return A, B, A + t * direcao(alfa)

## Básico

**Exercício 1.** Identificar o caso de congruência (ou "nenhum") em cada par de triângulos.

In [2]:
# Em cada item, o truque é olhar ONDE está cada elemento dado.
# (a) AB, B̂, BC: o ângulo B̂ é formado pelos lados AB e BC (fica entre eles)  -> LAL
# (b) Â, AB, B̂: o lado AB liga os vértices dos dois ângulos (fica entre eles) -> ALA
# (c) os três lados                                                           -> LLL
# (d) só ângulos: triângulos ampliados/reduzidos têm os mesmos ângulos        -> nenhum (AAA)
# (e) AB, Â, Ĉ: Â encosta em AB (adjacente), Ĉ fica de frente para AB (oposto) -> LAAo
# (f) AB, BC, Â: Â NÃO fica entre AB e BC. É um LLA, e o lado oposto a Â (BC = 4) é
#     o MENOR dos dois; pelo widget da seção 2 (AB = 6, Â = 30°, BC = 4), existem dois
#     triângulos diferentes com esses dados                                   -> nenhum
casos_ex1 = ["LAL", "ALA", "LLL", "nenhum", "LAAo", "nenhum"]
print(casos_ex1)

['LAL', 'ALA', 'LLL', 'nenhum', 'LAAo', 'nenhum']


**Exercício 2.** Elementos correspondentes a partir de $\triangle PQR \equiv \triangle LMN$.

In [3]:
# A ordem das letras diz a correspondência: P↔L, Q↔M, R↔N (1ª↔1ª, 2ª↔2ª, 3ª↔3ª).
# (a) Cada lado é formado por duas letras: troque cada uma pela sua correspondente.
#     PQ -> LM, QR -> MN, RP -> NL
lados_ex2 = {"PQ": "LM", "QR": "MN", "RP": "NL"}
angulos_ex2 = {"P": "L", "Q": "M", "R": "N"}

# (b) Elementos correspondentes de triângulos congruentes são congruentes:
#     LM ↔ PQ, então LM = 7 cm; M̂ ↔ Q̂, então M̂ = 48°.
medida_LM_ex2 = 7
medida_M_ex2 = 48
#     RP ↔ NL, então é o lado NL (ou LN, o mesmo segmento) que mede 9 cm.
lado_de_9cm_ex2 = "NL"

print(lados_ex2, angulos_ex2)
print(medida_LM_ex2, medida_M_ex2, lado_de_9cm_ex2)

{'PQ': 'LM', 'QR': 'MN', 'RP': 'NL'} {'P': 'L', 'Q': 'M', 'R': 'N'}
7 48 NL


## Intermediário

**Exercício 3.** Função que decide, pelas coordenadas, se dois triângulos são congruentes.

In [4]:
def sao_congruentes_ex3(triangulo_1, triangulo_2) -> bool:
    """True se os dois triângulos (listas de três vértices) forem congruentes."""
    # Pelo caso LLL, basta que os três lados sejam congruentes.
    # Como os vértices podem vir em qualquer ordem, não dá para comparar "AB com DE":
    # ordenamos as três medidas de cada triângulo e comparamos menor com menor,
    # médio com médio e maior com maior.
    def lados_ordenados(triangulo):
        P, Q, R = triangulo
        return sorted([distancia(P, Q), distancia(Q, R), distancia(R, P)])

    return all(math.isclose(x, y) for x, y in zip(lados_ordenados(triangulo_1), lados_ordenados(triangulo_2)))


modelo = [(0, 0), (4, 0), (1, 3)]
print(sao_congruentes_ex3(modelo, [(1, 3), (0, 0), (4, 0)]))   # mesmos vértices, outra ordem: True
print(sao_congruentes_ex3(modelo, [(0, 0), (-4, 0), (-1, 3)]))  # refletido: True
print(sao_congruentes_ex3(modelo, [(0, 0), (8, 0), (2, 6)]))    # ampliado (mesmos ângulos!): False

True
True
False


**Exercício 4.** Dois triângulos retângulos com hipotenusa e um cateto congruentes.

In [5]:
R1, P1, Q1 = (0, 0), (5, 0), (0, 12)
R2, P2, Q2 = (10, 2), (13, 6), (0.4, 9.2)

# (a) O vértice do ângulo vai no 1º argumento; os outros dois indicam os lados.
angulo_R1_ex4 = medir_angulo(R1, P1, Q1)
angulo_R2_ex4 = medir_angulo(R2, P2, Q2)

# (b) Os catetos são os lados que formam o ângulo reto: R1P1 (já conhecido, 5) e R1Q1.
outro_cateto_t1_ex4 = distancia(R1, Q1)
outro_cateto_t2_ex4 = distancia(R2, Q2)

# (c) Agora os três lados batem: 5, 12 e 13 nos dois triângulos -> LLL.
#     (Também valeria LAL: cateto, ângulo reto, cateto.)
caso_ex4 = "LLL"

# (d) O ângulo de 90° é oposto à hipotenusa, que é o MAIOR lado do triângulo retângulo
#     (ao maior ângulo opõe-se o maior lado, 0404a). É exatamente a situação especial em que
#     o "LLA" funciona: ângulo conhecido oposto ao maior dos dois lados conhecidos.
lado_oposto_ao_reto_ex4 = "hipotenusa"

print(f"{angulo_R1_ex4:.6f}  {angulo_R2_ex4:.6f}")
print(f"{outro_cateto_t1_ex4:.6f}  {outro_cateto_t2_ex4:.6f}")
print(f"hipotenusas: {distancia(P1, Q1):.6f}  {distancia(P2, Q2):.6f}")
print(caso_ex4, lado_oposto_ao_reto_ex4)

90.000000  90.000000
12.000000  12.000000
hipotenusas: 13.000000  13.000000
LLL hipotenusa


## Desafio

**Exercício 5.** A diagonal do paralelogramo e os lados opostos congruentes.

In [6]:
A5, B5, C5, D5 = np.array([0.0, 0.0]), np.array([6.0, 0.0]), np.array([8.0, 3.0]), np.array([2.0, 3.0])

# (a) Em "BÂC", a letra do meio (A) é o vértice: medir_angulo(A, B, C).
angulo_1_ex5 = medir_angulo(A5, B5, C5)  # BÂC
angulo_2_ex5 = medir_angulo(C5, D5, A5)  # DĈA
angulo_3_ex5 = medir_angulo(C5, B5, A5)  # BĈA
angulo_4_ex5 = medir_angulo(A5, D5, C5)  # DÂC
print(f"ângulo 1 = {angulo_1_ex5:.4f}°, ângulo 2 = {angulo_2_ex5:.4f}°")
print(f"ângulo 3 = {angulo_3_ex5:.4f}°, ângulo 4 = {angulo_4_ex5:.4f}°")

# (b) Em △ABC: ângulo 1 (em A), lado AC, ângulo 3 (em C).
#     Em △CDA: ângulo 2 (em C), lado CA, ângulo 4 (em A).
#     Dois ângulos e o lado ENTRE eles (AC liga os dois vértices) -> ALA.
caso_ex5 = "ALA"

# (c) Emparelhando pelos ângulos congruentes:
#     ângulo 1 (em A de △ABC) ≡ ângulo 2 (em C de △CDA)  -> A ↔ C
#     ângulo 3 (em C de △ABC) ≡ ângulo 4 (em A de △CDA)  -> C ↔ A
#     sobra B ↔ D
#     Escrevendo na ordem de A, B, C: △ABC ≡ △CDA
correspondencia_ex5 = "CDA"

# (d) AB ocupa as posições 1 e 2 de "ABC" -> posições 1 e 2 de "CDA": CD
#     BC ocupa as posições 2 e 3 de "ABC" -> posições 2 e 3 de "CDA": DA
lado_de_AB_ex5 = "CD"
lado_de_BC_ex5 = "DA"
print(caso_ex5, correspondencia_ex5, lado_de_AB_ex5, lado_de_BC_ex5)

# Conferindo: os lados opostos têm mesmo a mesma medida.
print(f"AB = {distancia(A5, B5):.4f} e CD = {distancia(C5, D5):.4f}")
print(f"BC = {distancia(B5, C5):.4f} e DA = {distancia(D5, A5):.4f}")
# Por que os ângulos 1 e 2 (e 3 e 4) são congruentes em QUALQUER paralelogramo? Eles são
# "alternos internos" entre retas paralelas, assunto do notebook de Paralelismo.

ângulo 1 = 20.5560°, ângulo 2 = 20.5560°
ângulo 3 = 35.7539°, ângulo 4 = 35.7539°
ALA CDA CD DA
AB = 6.0000 e CD = 6.0000
BC = 3.6056 e DA = 3.6056


**Exercício 6.** Reconstrução de um triângulo parcialmente apagado com o caso ALA.

In [7]:
# (a) Soma dos ângulos internos: 180 − 50 − 60 = 70
angulo_C_ex6 = 180 - 50 - 60

# (b) Um lado e os dois ângulos das pontas determinam o triângulo (ALA):
#     basta seguir a receita e medir o que ficou faltando.
A6, B6, C6 = construir_por_angulos(50, 60, base=8)
lado_AC_ex6 = distancia(A6, C6)
lado_BC_ex6 = distancia(B6, C6)

# (c) DE = AB, D̂ = Â, Ê = B̂: dois ângulos e o lado entre eles -> ALA.
caso_ex6 = "ALA"

# (d) Como △DEF ≡ △ABC (D↔A, E↔B, F↔C), TODOS os elementos correspondentes são congruentes.
#     DF corresponde a AC, então DF = AC, sem precisar medir.
df_igual_ac_ex6 = True

print(angulo_C_ex6, f"AC = {lado_AC_ex6:.4f}", f"BC = {lado_BC_ex6:.4f}", caso_ex6, df_igual_ac_ex6)
# Repare: o maior ângulo (Ĉ = 70°) está oposto ao maior lado (AB = 8), e o menor (Â = 50°)
# está oposto ao menor lado (BC ≈ 6,52), como em 0404a.

70 AC = 7.3728 BC = 6.5217 ALA True
